In [1]:
import yfinance as yf
import pandas as pd
import os

In [2]:
# Daily USD/PHP from 2018 to today
ovx = yf.download("^OVX", start="2015-01-01", end="2026-05-04", 
                 interval="1d", auto_adjust=False, multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [3]:
ovx

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2015-01-02,54.250000,54.250000,54.250000,50.240002,50.240002,0
2015-01-05,57.669998,57.669998,59.250000,56.660000,57.040001,0
2015-01-06,56.450001,56.450001,59.889999,55.509998,57.669998,0
2015-01-07,53.250000,53.250000,55.689999,51.840000,55.410000,0
2015-01-08,50.959999,50.959999,53.250000,49.889999,53.250000,0
...,...,...,...,...,...,...
2026-04-27,73.110001,73.110001,76.959999,73.110001,76.160004,0
2026-04-28,70.500000,70.500000,74.440002,70.449997,74.430000,0
2026-04-29,75.959999,75.959999,75.959999,71.800003,72.169998,0


In [4]:
ovx.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2849 entries, 2015-01-02 to 2026-05-01
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  2849 non-null   float64
 1   Close      2849 non-null   float64
 2   High       2849 non-null   float64
 3   Low        2849 non-null   float64
 4   Open       2849 non-null   float64
 5   Volume     2849 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 155.8 KB


In [6]:
ovx.isnull().sum()

Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

In [7]:
df_ovx= ovx[["Close"]].rename(columns={"Close": "ovx"})

In [8]:
df_ovx

,ovx
Date,
2015-01-02,54.250000
2015-01-05,57.669998
2015-01-06,56.450001
2015-01-07,53.250000
2015-01-08,50.959999
...,...
2026-04-27,73.110001
2026-04-28,70.500000
2026-04-29,75.959999


In [9]:
df_ovx = df_ovx.reset_index() # date is alr in datetime
df_ovx.columns = ["date", "ovx"]
df_ovx["date"] = pd.to_datetime(df_ovx["date"])

In [10]:
df_ovx = df_ovx.set_index("date").sort_index()

In [11]:
df_ovx = df_ovx.resample("W-TUE").agg(
    ovx_close=("ovx", "last"),
    ovx_mean=("ovx", "mean")
)
# weekly.index = weekly.index + pd.Timedelta(days=3)
# weekly.index.name = "week_tuesday"

In [12]:
df_ovx

,ovx_close,ovx_mean
date,,
2015-01-06,56.450001,56.123333
2015-01-13,53.779999,52.238000
2015-01-20,57.990002,56.342500
2015-01-27,54.230000,55.998000
2015-02-03,59.680000,57.202000
...,...,...
2026-04-07,98.790001,94.967501
2026-04-14,75.349998,79.912001
2026-04-21,85.570000,74.728001


In [13]:
df_ovx.to_parquet("../data/semi-cleaned/weekly_ovx.parquet")